<a href="https://colab.research.google.com/github/Ign4cho/Ign4cho.github.io/blob/main/WhisperVideoDrive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

If you're looking at this on GitHub and new to Python Notebooks or Colab, click the Google Colab badge above 👆

#📼 OpenAI Whisper + Google Drive Video Transcription

📺 Getting started video: https://youtu.be/YGpYinji7II

###This application will extract audio from all the video files in a Google Drive folder and create a high-quality transcription with OpenAI's Whisper automatic speech recognition system.

*Note: This requires giving the application permission to connect to your drive. Only you will have access to the contents of your drive, but please read the warnings carefully.*

This notebook application:
1. Connects to your Google Drive when you give it permission.
2. Creates a WhisperVideo folder and three subfolders (ProcessedVideo, AudioFiles and TextFiles.)
3. When you run the application it will search for all the video files (.mp4, .mov, mkv and .avi) in your WhisperVideo folder, transcribe them and then move the file to WhisperVideo/ProcessedVideo and save the transcripts to WhisperVideo/TextFiles. It will also add a copy of the new audio file to WhisperVideo/AudioFiles

###**For faster performance set your runtime to "GPU"**
*Click on "Runtime" in the menu and click "Change runtime type". Select "GPU".*


**Note: If you add a new file after running this application you'll need to remount the drive in step 1 to make them searchable**

##1. Load the code libraries

In [1]:
!pip install git+https://github.com/openai/whisper.git
!sudo apt update && sudo apt install ffmpeg
!pip install librosa

import whisper
import time
import librosa
import soundfile as sf
import re
import os

# model = whisper.load_model("tiny")
# model = whisper.load_model("base")
# model = whisper.load_model("small")
model = whisper.load_model("medium") # Cargamos el modelo medium multilingüe
# model = whisper.load_model("large")

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-rz4943ug
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-rz4943ug
  Resolved https://github.com/openai/whisper.git to commit 86098128c0b4f24f0e2aa2994de830614b474227
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=804248 sha256=768a88135150947b448162f53ea8b7e8efbadcc997d98388841a74e8836db55b
  Stored in directory: /tmp/pip-ephem-wheel-cache-r0u_u_dk/wheels/0e/80/9c/02c93e9c61634842951034da68bec46cf781018d5b53bb239c
Successfully built openai-whisper
Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [4,685 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InR

100%|█████████████████████████████████████| 1.42G/1.42G [00:22<00:00, 66.9MiB/s]


##2. Give the application permission to mount the drive and create the folders

In [20]:
import os

# === VARIABLES CONFIGURABLES ===
id_video = "1J-tykrtYA84gy8LW_4D7XyWiOYp5vLE0"  # Reemplaza con el ID de tu próximo video
nombre_clase = "clase8-p2-teoria-integrateandfire.mp4"                   # Ponle el nombre que quieras (debe terminar en .mp4)
# ===============================

# 1. Creamos las carpetas en el almacenamiento temporal de Colab (/content/)
folders =  ["WhisperVideo/", "WhisperVideo/ProcessedVideo/", "WhisperVideo/TextFiles/", "WhisperVideo/AudioFiles/"]
for folder in folders:
  path = "/content/" + folder
  if not os.path.exists(path):
    os.mkdir(path)

# 2. Descargamos el video de Drive usando las variables
ruta_destino = f"/content/WhisperVideo/{nombre_clase}"
!gdown --id {id_video} -O "{ruta_destino}"

print(f"¡Carpetas creadas y video '{nombre_clase}' descargado con éxito!")

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1J-tykrtYA84gy8LW_4D7XyWiOYp5vLE0
From (redirected): https://drive.google.com/uc?id=1J-tykrtYA84gy8LW_4D7XyWiOYp5vLE0&confirm=t&uuid=a146f190-4cf4-4515-a537-497bf6904904
To: /content/WhisperVideo/clase8-p2-teoria-integrateandfire.mp4
100% 662M/662M [00:08<00:00, 74.7MB/s]
¡Carpetas creadas y video 'clase8-p2-teoria-integrateandfire.mp4' descargado con éxito!


##3. Upload any video files you want transcribed in the "WhisperVideo" folder in your Google Drive.

##4. Extract audio from the video files and create a transcription

In [21]:
import os
import librosa
import soundfile as sf
from google.colab import files # Para descargar el archivo al final

# Obtenemos la lista de archivos de video desde la carpeta temporal
video_files = os.listdir("/content/WhisperVideo/")

# Recorremos los videos (en este caso será el que acabas de descargar)
for video_file in video_files:

  # Omitir si no es un formato de video
  if not video_file.endswith((".mp4", ".mov", ".avi", ".mkv")):
    continue

  # Rutas a la memoria de Colab
  video_path = "/content/WhisperVideo/" + video_file
  audio_path = "/content/WhisperVideo/AudioFiles/" + video_file[:-4] + ".wav"

  print(f"Extrayendo audio de {video_file}...")
  y, sr = librosa.load(video_path, sr=16000) # Carga el audio a 16 kHz
  sf.write(audio_path, y, sr) # Guarda como .wav

  print("Transcribiendo con Whisper (esto puede tardar unos minutos)...")
  # Transcribe el audio file usando Whisper
  result = model.transcribe(audio_path)
  text = result["text"].strip()
  text = text.replace(". ", ".\n\n")

  # Guarda la transcripción como archivo de texto
  text_file = video_file[:-4] + ".txt"
  text_path = "/content/WhisperVideo/TextFiles/" + text_file
  with open(text_path, "w") as f:
    f.write(text)

  # Mueve el video procesado
  processed_path = "/content/WhisperVideo/ProcessedVideo/" + video_file
  os.rename(video_path, processed_path)

  print(f"¡Listo! Procesado {video_file}. Guardado como {text_file}")

  # Descarga el archivo .txt a tu PC automáticamente
  files.download(text_path)

Extrayendo audio de clase8-p2-teoria-integrateandfire.mp4...


/tmp/ipykernel_2285/3390484821.py:21: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=16000) # Carga el audio a 16 kHz
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Transcribiendo con Whisper (esto puede tardar unos minutos)...
¡Listo! Procesado clase8-p2-teoria-integrateandfire.mp4. Guardado como clase8-p2-teoria-integrateandfire.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>